In [1]:
import os
os.environ["PYTHONNOUSERSITE"] = "1"

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoConfig, BitsAndBytesConfig
from adaptive_snapkv.monkeypatch.monkeypatch import replace_llama_adaptive

# replace_llama_adaptive()

model_id = "meta-llama/Meta-Llama-3-8B"
config = AutoConfig.from_pretrained(model_id, trust_remote_code=True)

# AdaKV settings (same as before)
config.window_size = 32
config.base_capacity = 512
config.kernel_size = 7
config.pooling = "maxpool"
config.floor_alpha = 0.5
config.pyram_mode = False
config.pyram_beta = 20
# Missing attributes required by AdaKV
config.skip = False
config.normalize = False
config.gqa_support = True
config.gqa_func = "mean"   # 'max' or 'mean'
config.base_capacity = 2048


quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    config=config,
    quantization_config=quantization_config,
    attn_implementation="flash_attention_2",   # official AdaKV uses flash_attn
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

# No need to pass device_map – the 4-bit model will load on GPU 0 automatically

/home/kashora/miniconda3/envs/adakv_final/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`low_cpu_mem_usage` was None, now set to True since model is quantized.
Loading checkpoint shards: 100%|██████████| 4/4 [01:29<00:00, 22.43s/it]


In [2]:

from datasets import load_dataset
from tqdm import tqdm
print("Loading WikiText-2 validation segment...")
test_dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="validation")
eval_text = "\n\n".join([text for text in test_dataset["text"] if len(text.strip()) > 0])[:35000]
print(f"Dataset loaded. Character count: {len(eval_text)}")


MAX_LENGTH = 1024 * 2
STRIDE = MAX_LENGTH // 2


Loading WikiText-2 validation segment...
Dataset loaded. Character count: 35000


In [12]:

prompt = "Explain the theory of general relativity in simple terms."
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")



In [13]:
import time
import torch

def evaluate_generation(model, tokenizer, prompt, max_new_tokens=100):
    input = tokenizer(prompt, truncation=False, return_tensors="pt").to("cuda")
    context_length = input.input_ids.shape[-1]

    torch.cuda.synchronize()
    t0 = time.time()
    output = model.generate(
        **input,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=1.0,
        eos_token_id=[tokenizer.eos_token_id],
    )[0]
    torch.cuda.synchronize()
    t = time.time() - t0

    generated = tokenizer.decode(output[context_length:], skip_special_tokens=True)
    tokens_per_sec = max_new_tokens / t
    return generated, t, tokens_per_sec


In [21]:
import json, time, torch, gc
from datasets import load_dataset
from tqdm import tqdm

def evaluate_longbench(model, tokenizer, dataset_name="qasper", 
                       max_length=4096, max_new_tokens=128, out_path=None):
    """Run one LongBench dataset on the already-loaded model."""
    dataset2prompt = json.load(open("experiments/LongBench/config/dataset2prompt.json"))
    dataset2maxlen = json.load(open("experiments/LongBench/config/dataset2maxlen.json"))
    
    data = load_dataset("THUDM/LongBench", dataset_name, split="test")
    prompt_format = dataset2prompt[dataset_name]
    max_gen = dataset2maxlen[dataset_name]
    
    preds = []
    for json_obj in tqdm(data):
        prompt = prompt_format.format(**json_obj)
        tokenized = tokenizer(prompt, truncation=False, return_tensors="pt").input_ids[0]
        
        if len(tokenized) > max_length:
            half = max_length // 2
            prompt = (tokenizer.decode(tokenized[:half], skip_special_tokens=True) + 
                      tokenizer.decode(tokenized[-half:], skip_special_tokens=True))
        
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        ctx_len = inputs.input_ids.shape[-1]
        
        with torch.no_grad():
            output = model.generate(
                **inputs, max_new_tokens=max_gen,
                do_sample=False, temperature=1.0,
                eos_token_id=[tokenizer.eos_token_id],
            )[0]
        
        pred = tokenizer.decode(output[ctx_len:], skip_special_tokens=True)
        preds.append({"pred": pred, "answers": json_obj["answers"]})
        
        gc.collect()
        torch.cuda.empty_cache()
    
    if out_path:
        with open(out_path, "w") as f:
            for p in preds:
                json.dump(p, f, ensure_ascii=False)
                f.write("\n")
    
    return preds

In [23]:


import transformers.models.llama.modeling_llama as llama_models
_vanilla_model_forward = llama_models.LlamaModel.forward
_vanilla_attn_forward = llama_models.LlamaFlashAttention2.forward

import importlib
import types
import transformers.models.llama.modeling_llama as llama_models

import gc

# Reload to restore original class methods
llama_models = importlib.reload(llama_models)

# Swap model to vanilla
model.model.forward = types.MethodType(
    llama_models.LlamaModel.forward, model.model
)

# Swap ALL attention layer forwards
for layer in model.model.layers:
    attn = layer.self_attn
    # Restore vanilla class method
    attn.forward = types.MethodType(
        llama_models.LlamaFlashAttention2.forward, attn
    )


out_vanilla = model.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(out_vanilla[0]))

base_gen_text, base_gen_time, base_tok_s = evaluate_generation(model, tokenizer, prompt)
print(f"Generated ({base_tok_s:.1f} tok/s):\n{base_gen_text}")

base_pred = evaluate_longbench(model, tokenizer, dataset_name="qasper", max_length=4096, max_new_tokens=128, out_path="qasper_preds.jsonl")

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


<|begin_of_text|>Explain the theory of general relativity in simple terms. What is the significance of this theory? What is the difference between Einstein’s theory of relativity and Newton’s theory of gravitation?
Newton’s theory of gravitation
The theory of gravitation is a scientific theory that explains the force of attraction between
Generated (21.8 tok/s):
 What are the implications of this theory? What are the implications of this theory? What are the implications of this theory? What are the implications of this theory? What are the implications of this theory? What are the implications of this theory? What are the implications of this theory? What are the implications of this theory? What are the implications of this theory? What are the implications of this theory? What are the implications of this theory? What are the implications of this theory? What are the implications


/home/kashora/miniconda3/envs/adakv_final/lib/python3.10/site-packages/datasets/load.py:1491: FutureWarning: The repository for THUDM/LongBench contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/THUDM/LongBench
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(
100%|██████████| 200/200 [22:53<00:00,  6.87s/it]


In [24]:
import types
import torch
from transformers.cache_utils import Cache as HFCache
import adaptive_snapkv.monkeypatch.adaptive_llama_hijack as alh
import transformers.models.llama.modeling_llama as llama_models


import adaptive_snapkv.monkeypatch.adaptive_llama_hijack as alh

model.model.forward = types.MethodType(
    alh.adaptive_LlamaModel_forward, model.model
)
for layer in model.model.layers:
    attn = layer.self_attn
    attn.forward = types.MethodType(
        alh.adaptive_llama_flash_attn2_forward, attn
    )


print("✓ Cache init fix applied")


out_ada = model.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(out_ada[0]))

ada_gen_text, ada_gen_time, ada_tok_s = evaluate_generation(model, tokenizer, prompt)
print(f"Generated ({ada_tok_s:.1f} tok/s):\n{ada_gen_text}")

ada_pred = evaluate_longbench(model, tokenizer, dataset_name="qasper", max_length=4096, max_new_tokens=128, out_path="qasper_preds_ada.jsonl")

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


✓ Cache init fix applied


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


<|begin_of_text|>Explain the theory of general relativity in simple terms. Albert Einstein’s theory of general relativity has been hailed as one of the greatest intellectual achievements of the 20th century. But what is it, exactly? The theory of general relativity, which Einstein developed between 1907 and 1915
Generated (14.8 tok/s):
 What are the implications of this theory? What are the implications of this theory? What are the implications of this theory? What are the implications of this theory? What are the implications of this theory? What are the implications of this theory? What are the implications of this theory? What are the implications of this theory? What are the implications of this theory? What are the implications of this theory? What are the implications of this theory? What are the implications of this theory? What are the implications


100%|██████████| 200/200 [19:51<00:00,  5.96s/it]


In [28]:
import re, string, json, os, numpy as np
from collections import Counter
from rouge import Rouge

def normalize_answer(s):
    def remove_articles(text):
        return re.sub(r"\b(a|an|the)\b", " ", text)
    def white_space_fix(text):
        return " ".join(text.split())
    def remove_punc(text):
        return "".join(ch for ch in text if ch not in set(string.punctuation))
    def lower(text):
        return text.lower()
    return white_space_fix(remove_articles(remove_punc(lower(s))))

def f1_score(prediction, ground_truth):
    common = Counter(prediction) & Counter(ground_truth)
    num_same = sum(common.values())
    if num_same == 0: return 0
    precision = 1.0 * num_same / len(prediction)
    recall = 1.0 * num_same / len(ground_truth)
    return (2 * precision * recall) / (precision + recall)

def qa_f1_score(prediction, ground_truth, **kwargs):
    return f1_score(normalize_answer(prediction).split(), normalize_answer(ground_truth).split())

def rouge_score(prediction, ground_truth, **kwargs):
    try:
        return Rouge().get_scores([prediction], [ground_truth], avg=True)["rouge-l"]["f"]
    except:
        return 0.0

dataset2metric = {
    "narrativeqa": qa_f1_score, "qasper": qa_f1_score,
    "multifieldqa_en": qa_f1_score, "hotpotqa": qa_f1_score,
    "2wikimqa": qa_f1_score, "musique": qa_f1_score,
    "gov_report": rouge_score, "qmsum": rouge_score,
    "multi_news": rouge_score, "samsum": rouge_score,
}

def score_dataset(jsonl_path):
    with open(jsonl_path) as f:
        data = [json.loads(line) for line in f]
    dataset = os.path.splitext(os.path.basename(jsonl_path))[0]
    metric = dataset2metric.get(dataset, qa_f1_score)
    scores = []
    for item in data:
        best = 0
        for gt in item["answers"]:
            best = max(best, metric(item["pred"], gt))
        scores.append(best)
    return dataset, round(100 * np.mean(scores), 2)



In [31]:
import json, time, torch, gc, os
from datasets import load_dataset
from tqdm import tqdm

dataset2prompt = json.load(open("experiments/LongBench/config/dataset2prompt.json"))
dataset2maxlen = json.load(open("experiments/LongBench/config/dataset2maxlen.json"))

datasets = ["narrativeqa", "qasper", "hotpotqa", "gov_report", "multi_news", "samsum"]
max_length = 4096

def get_preds(model, tokenizer, dataset_name, out_path):
    data = load_dataset("THUDM/LongBench", dataset_name, split="test")
    prompt_format = dataset2prompt[dataset_name]
    max_gen = dataset2maxlen[dataset_name]

    preds, times = [], []
    for json_obj in tqdm(data, desc=dataset_name):
        prompt = prompt_format.format(**json_obj)
        tokenized = tokenizer(prompt, truncation=False, return_tensors="pt").input_ids[0]
        if len(tokenized) > max_length:
            half = max_length // 2
            prompt = (tokenizer.decode(tokenized[:half], skip_special_tokens=True) +
                      tokenizer.decode(tokenized[-half:], skip_special_tokens=True))

        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        ctx_len = inputs.input_ids.shape[-1]

        torch.cuda.synchronize()
        t0 = time.time()
        with torch.no_grad():
            output = model.generate(
                **inputs, max_new_tokens=max_gen,
                do_sample=False, temperature=1.0,
                eos_token_id=[tokenizer.eos_token_id],
            )[0]
        torch.cuda.synchronize()
        dt = time.time() - t0
        times.append(dt)

        pred = tokenizer.decode(output[ctx_len:], skip_special_tokens=True)
        preds.append({"pred": pred, "answers": json_obj["answers"],
                      "all_classes": json_obj["all_classes"], "length": json_obj["length"]})

        gc.collect()
        torch.cuda.empty_cache()

    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, "w") as f:
        for p in preds:
            json.dump(p, f, ensure_ascii=False)
            f.write("\n")

    return np.mean(times)

In [32]:
print("=== Vanilla ===")
# swap to vanilla (reload + re-patch)
import importlib, types
import transformers.models.llama.modeling_llama as llama_models
llama_models = importlib.reload(llama_models)
model.model.forward = types.MethodType(llama_models.LlamaModel.forward, model.model)
for layer in model.model.layers:
    c = layer.self_attn.__class__
    vanilla_fwd = getattr(llama_models, c.__name__).forward
    layer.self_attn.forward = types.MethodType(vanilla_fwd, layer.self_attn)

for ds in datasets:
    get_preds(model, tokenizer, ds, f"pred/vanilla_run/{ds}.jsonl")

# ── AdaKV run ──
print("\n=== AdaKV ===")
import adaptive_snapkv.monkeypatch.adaptive_llama_hijack as alh
alh = importlib.reload(alh)
llama_models.LlamaModel.forward = alh.adaptive_LlamaModel_forward
model.model.forward = types.MethodType(alh.adaptive_LlamaModel_forward, model.model)
for layer in model.model.layers:
    c = layer.self_attn.__class__
    ada_fwd = getattr(alh, f"adaptive_{c.__name__.lower()}_forward",
                     alh.adaptive_llama_flash_attn2_forward)
    layer.self_attn.forward = types.MethodType(ada_fwd, layer.self_attn)

for ds in datasets:
    get_preds(model, tokenizer, ds, f"pred/adakv_run/{ds}.jsonl")

print("\nDone! Results in pred/vanilla_run/ and pred/adakv_run/")

=== Vanilla ===


/home/kashora/miniconda3/envs/adakv_final/lib/python3.10/site-packages/datasets/load.py:1491: FutureWarning: The repository for THUDM/LongBench contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/THUDM/LongBench
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(
Generating test split: 200 examples [00:00, 391.66 examples/s]
narrativeqa:   0%|          | 0/200 [00:00<?, ?it/s]/home/kashora/miniconda3/envs/adakv_final/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:606: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id


=== AdaKV ===


samsum: 100%|██████████| 200/200 [19:42<00:00,  5.91s/it]


Done! Results in pred/vanilla_run/ and pred/adakv_run/


In [ ]:
import os

vanilla_dir = "pred/vanilla_run/"
adakv_dir = "pred/adakv_run/"

results = []
for fname in os.listdir(vanilla_dir):
    if not fname.endswith(".jsonl"):
        continue
    
    ds = fname.replace(".jsonl", "")
    van_path = os.path.join(vanilla_dir, fname)
    ada_path = os.path.join(adakv_dir, fname)
    
    _, van_score = score_dataset(van_path)
    
    if os.path.exists(ada_path):
        _, ada_score = score_dataset(ada_path)
        delta = ada_score - van_score
    else:
        ada_score = None
        delta = None
    
    results.append((ds, van_score, ada_score, delta))

# Print comparison table
print(f"{'Dataset':<20} {'Vanilla':>8} {'AdaKV':>8} {'Δ':>8}")
print("-" * 48)
van_scores = []
for ds, v, a, d in sorted(results, key=lambda x: x[3] if x[3] is not None else 0):
    van_scores.append(v)
    a_str = f"{a:>8.2f}" if a is not None else "   N/A  "
    d_str = f"{d:>+8.2f}" if d is not None else "   N/A  "
    print(f"{ds:<20} {v:>8.2f} {a_str} {d_str}")

print("-" * 48)
avg_v = np.mean(van_scores)
completed = [r for r in results if r[2] is not None]
avg_a = np.mean([r[2] for r in completed]) if completed else 0
print(f"{'AVERAGE':<20} {avg_v:>8.2f} {'N/A':>8}" if not completed else
      f"{'AVERAGE':<20} {avg_v:>8.2f} {avg_a:>8.2f} {avg_a-avg_v:>+8.2f}")